In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.optim import Adam
from torch.utils.data import DataLoader
from source.version3.data import trainLoader
from source.version3.model import EfficientModel
from source.version3.train import trainModel
from source.version3.loss import bi_tempered_logistic_loss
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts as CosLR

In [3]:
def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(2017)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/raw/train_images/'
    loader['label_path'] = '../../data/raw/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 7
    params['num_workers'] = 3
    params['drop_last'] = True
    train = DataLoader(train, **params, shuffle=True)
    valid = DataLoader(valid, **params, shuffle=False)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = Adam(model.parameters(), lr=1e-04, weight_decay=1e-6)
    schedular = CosLR(optimizer, T_0=10, T_mult=1, eta_min=1e-6, last_epoch=-1)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = bi_tempered_logistic_loss
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version3/model_{}.pt'.format(fold)
    trainer['epochs'] = 10
    trainer['batch'] = 7
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [5]:
train(0)

Train Images: 17117 Valid Images: 4280


100% 17115/17115 [13:46<00:00, 20.71it/s, trn_ls=0.3458, val_ls=0.2237, val_mt=0.8377]
100% 17115/17115 [13:43<00:00, 20.78it/s, trn_ls=0.2310, val_ls=0.1843, val_mt=0.8702]
100% 17115/17115 [13:44<00:00, 20.76it/s, trn_ls=0.2092, val_ls=0.1747, val_mt=0.8773]
100% 17115/17115 [13:40<00:00, 20.87it/s, trn_ls=0.1913, val_ls=0.1794, val_mt=0.8719]
100% 17115/17115 [13:40<00:00, 20.87it/s, trn_ls=0.1844, val_ls=0.1862, val_mt=0.8621]
100% 17115/17115 [13:39<00:00, 20.89it/s, trn_ls=0.1739, val_ls=0.1646, val_mt=0.8840]
100% 17115/17115 [13:42<00:00, 20.81it/s, trn_ls=0.1677, val_ls=0.1640, val_mt=0.8822]
100% 17115/17115 [13:39<00:00, 20.88it/s, trn_ls=0.1613, val_ls=0.1668, val_mt=0.8801]
100% 17115/17115 [13:39<00:00, 20.88it/s, trn_ls=0.1577, val_ls=0.1658, val_mt=0.8810]
100% 17115/17115 [13:40<00:00, 20.85it/s, trn_ls=0.1576, val_ls=0.1692, val_mt=0.8794]


In [ ]:
train(1)

Train Images: 17117 Valid Images: 4280


 39% 6748/17115 [05:05<08:07, 21.25it/s, trn_ls=0.43450]

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)